# MV Generator - MuseTalk Colab MVP

This notebook runs a self-managed MuseTalk inference flow in Google Colab.

Goal:

```text
base MP4 + MP3 -> MuseTalk -> clean lip-sync MP4
```

Use only media you own or have permission to process. AI-generated assets still depend on the generation platform terms and any likeness/audio rights.

## 1. Runtime Check

In Colab, choose `Runtime -> Change runtime type -> GPU` before running the notebook.

In [ ]:
import os, sys, subprocess, textwrap, json, glob, shutil
print('Python:', sys.version)
print('GPU:')
!nvidia-smi || true
!ffmpeg -version | head -n 2 || true

## 2. Clone MuseTalk

In [ ]:
%cd /content
if not os.path.exists('/content/MuseTalk'):
    !git clone https://github.com/TMElyralab/MuseTalk.git
%cd /content/MuseTalk
!git rev-parse --short HEAD

## 3. Install Dependencies

Current Colab images may use Python 3.12, while MuseTalk's official dependency set expects Python 3.10-era wheels. This cell creates an isolated conda environment named `musetalk` with Python 3.10, then installs MuseTalk dependencies there instead of modifying Colab's built-in Python.

In [ ]:
%cd /content
!if [ ! -d /content/conda ]; then wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh && bash /tmp/miniforge.sh -b -p /content/conda; fi
import os
os.environ['PATH'] = '/content/conda/bin:' + os.environ['PATH']
!conda --version
!bash -lc 'source /content/conda/etc/profile.d/conda.sh && if ! conda env list | awk "{print \\$1}" | grep -qx musetalk; then conda create -y -n musetalk python=3.10; else echo "conda env musetalk already exists"; fi'
%cd /content/MuseTalk
!conda run -n musetalk python -m pip install -U "pip<25" "setuptools<70" wheel
!conda run -n musetalk python -m pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!conda run -n musetalk python -m pip install -r requirements.txt
!conda run -n musetalk python -m pip install --no-cache-dir -U openmim
!conda run -n musetalk mim install mmengine
!conda run -n musetalk mim install "mmcv==2.0.1"
!conda run -n musetalk mim install "mmdet==3.1.0"
!conda run -n musetalk mim install "mmpose==1.1.0"
!conda run -n musetalk python -c "import sys, torch, numpy; print('env python:', sys.version); print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available()); print('numpy:', numpy.__version__)"

## 4. Download Weights

Download model weights without relying on Hugging Face mirror defaults or deprecated `huggingface-cli` behavior. The first run can take a while.


In [ ]:
%cd /content/MuseTalk
import os
from pathlib import Path
from huggingface_hub import snapshot_download
import gdown

os.environ.pop('HF_ENDPOINT', None)

jobs = [
    ('TMElyralab/MuseTalk', 'models', ['musetalk/musetalk.json', 'musetalk/pytorch_model.bin', 'musetalkV15/musetalk.json', 'musetalkV15/unet.pth']),
    ('stabilityai/sd-vae-ft-mse', 'models/sd-vae', ['config.json', 'diffusion_pytorch_model.bin']),
    ('openai/whisper-tiny', 'models/whisper', ['config.json', 'pytorch_model.bin', 'preprocessor_config.json']),
    ('yzd-v/DWPose', 'models/dwpose', ['dw-ll_ucoco_384.pth']),
    ('ByteDance/LatentSync', 'models/syncnet', ['latentsync_syncnet.pt']),
]

for repo_id, local_dir, patterns in jobs:
    print('Downloading', repo_id, '->', local_dir)
    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        allow_patterns=patterns,
        local_dir_use_symlinks=False,
        resume_download=True,
    )

face_parse = 'models/face-parse-bisent/79999_iter.pth'
Path(face_parse).parent.mkdir(parents=True, exist_ok=True)
if not Path(face_parse).exists():
    gdown.download(
        url='https://drive.google.com/uc?id=154JgKpzCPW82qINcVieuPH3fZ2e0P812',
        output=face_parse,
        quiet=False,
    )

resnet = Path('models/face-parse-bisent/resnet18-5c106cde.pth')
if not resnet.exists():
    !curl -L https://download.pytorch.org/models/resnet18-5c106cde.pth -o models/face-parse-bisent/resnet18-5c106cde.pth

required = [
    'models/musetalkV15/unet.pth',
    'models/musetalkV15/musetalk.json',
    'models/dwpose/dw-ll_ucoco_384.pth',
    'models/syncnet/latentsync_syncnet.pt',
    'models/sd-vae/diffusion_pytorch_model.bin',
    'models/whisper/pytorch_model.bin',
    'models/face-parse-bisent/79999_iter.pth',
    'models/face-parse-bisent/resnet18-5c106cde.pth',
]
missing = [p for p in required if not Path(p).exists()]
print('missing:', missing)
assert not missing, missing
print('All required weights are present.')


## 5. Upload Your MVP Inputs

Upload these files from your local project:

- `outputs/gina_base_with_audio.mp4`
- `inputs/Café_no_Mar.mp3`

The notebook copies them into `/content/MuseTalk/data/mv_mvp/`.

In [ ]:
from google.colab import files
uploaded = files.upload()

work_dir = '/content/MuseTalk/data/mv_mvp'
os.makedirs(work_dir, exist_ok=True)

for name in uploaded.keys():
    src = f'/content/MuseTalk/{name}' if os.path.exists(f'/content/MuseTalk/{name}') else f'/content/{name}'
    if os.path.exists(name):
        src = name
    dst = os.path.join(work_dir, os.path.basename(name))
    shutil.copy(src, dst)
    print('copied', src, '->', dst)

print('Uploaded files:')
!find /content/MuseTalk/data/mv_mvp -maxdepth 1 -type f -printf '%f\n'

## 6. Normalize Inputs

MuseTalk recommends 25fps video. This cell converts the base video to 25fps H.264 and extracts a clean WAV from the MP3.

In [ ]:
%cd /content/MuseTalk

BASE_VIDEO = '/content/MuseTalk/data/mv_mvp/gina_base_with_audio.mp4'
BASE_AUDIO = '/content/MuseTalk/data/mv_mvp/Café_no_Mar.mp3'

# Fallback for files uploaded with normalized or renamed filenames.
if not os.path.exists(BASE_VIDEO):
    videos = glob.glob('/content/MuseTalk/data/mv_mvp/*.mp4')
    assert videos, 'No MP4 uploaded.'
    BASE_VIDEO = videos[0]
if not os.path.exists(BASE_AUDIO):
    audios = glob.glob('/content/MuseTalk/data/mv_mvp/*.mp3') + glob.glob('/content/MuseTalk/data/mv_mvp/*.wav')
    assert audios, 'No audio uploaded.'
    BASE_AUDIO = audios[0]

NORM_VIDEO = '/content/MuseTalk/data/mv_mvp/base_25fps.mp4'
NORM_AUDIO = '/content/MuseTalk/data/mv_mvp/audio_16k.wav'

!ffmpeg -y -i "$BASE_VIDEO" -an -vf fps=25,scale=1280:720,setsar=1 -c:v libx264 -pix_fmt yuv420p -movflags +faststart "$NORM_VIDEO"
!ffmpeg -y -i "$BASE_AUDIO" -ar 16000 -ac 1 "$NORM_AUDIO"
!ffprobe -v error -show_entries format=duration:stream=codec_type,codec_name,width,height,r_frame_rate,duration -of json "$NORM_VIDEO"
!ffprobe -v error -show_entries format=duration:stream=codec_type,codec_name,duration -of json "$NORM_AUDIO"

## 7. Write MuseTalk Inference Config

`bbox_shift` can strongly affect mouth-region placement. Start with `0`. If the lower face is poorly aligned, try `-7`, `-5`, `5`, or `7` and re-run inference.

In [ ]:
%cd /content/MuseTalk
BBOX_SHIFT = 0
CONFIG_PATH = '/content/MuseTalk/configs/inference/mv_mvp.yaml'
RESULT_DIR = '/content/MuseTalk/results/mv_mvp'
os.makedirs(os.path.dirname(CONFIG_PATH), exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

config_text = f'''task_0:\n  video_path: "{NORM_VIDEO}"\n  audio_path: "{NORM_AUDIO}"\n  bbox_shift: {BBOX_SHIFT}\n'''
with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    f.write(config_text)
print(config_text)

## 8. Run MuseTalk 1.5 Inference

This runs the official normal inference script with MuseTalk 1.5 weights. It can take several minutes for a 20-30 second clip on a free T4.


In [ ]:
%cd /content/MuseTalk
import os
os.environ['MPLBACKEND'] = 'Agg'
!PYTHONUNBUFFERED=1 MPLBACKEND=Agg conda run --no-capture-output -n musetalk python -m scripts.inference \
  --inference_config "$CONFIG_PATH" \
  --result_dir "$RESULT_DIR" \
  --unet_model_path models/musetalkV15/unet.pth \
  --unet_config models/musetalkV15/musetalk.json \
  --version v15
print('Result files:')
!find "$RESULT_DIR" -type f | sort


## 9. Merge Original Audio Back If Needed

Some inference scripts output video-only files. This cell finds the newest MP4 result and muxes the original uploaded audio back in.

In [ ]:
result_videos = sorted(glob.glob(RESULT_DIR + '/**/*.mp4', recursive=True), key=os.path.getmtime)
assert result_videos, 'No MuseTalk result MP4 found.'
RAW_RESULT = result_videos[-1]
FINAL_OUTPUT = '/content/musetalk_colab_final_with_audio.mp4'

print('Raw result:', RAW_RESULT)
!ffprobe -v error -show_entries stream=codec_type,codec_name -of json "$RAW_RESULT"
!ffmpeg -y -i "$RAW_RESULT" -i "$BASE_AUDIO" -map 0:v:0 -map 1:a:0 -c:v copy -c:a aac -shortest -movflags +faststart "$FINAL_OUTPUT"
!ffprobe -v error -show_entries format=duration,size:stream=index,codec_type,codec_name,width,height,r_frame_rate,duration -of json "$FINAL_OUTPUT"

## 10. Preview And Download

In [ ]:
from IPython.display import Video, display
display(Video(FINAL_OUTPUT, embed=True, width=960))
files.download(FINAL_OUTPUT)